# Main 03 — baseline discovery and sensitivity

Run nTRD-versus-TRD baseline regional, connectivity, MwC, and baseline-severity sensitivity analyses.

- Formal analysis universe: 12 cortical + 14 limbic ROIs (26 nodes)
- Cross-network connectivity: 12 × 14 = 168 edges
- Pipeline: `main`; step 3 of 10
- Run mode: standalone; all prerequisite steps load automatically in the same kernel.


In [1]:
from pathlib import Path
import os

_repo = Path(os.environ.get("METAFL_ROOT", Path.cwd())).resolve()
if not (_repo / "pipeline_steps").is_dir() and (_repo.parent / "pipeline_steps").is_dir():
    _repo = _repo.parent
if not (_repo / "pipeline_steps").is_dir():
    raise FileNotFoundError("Set METAFL_ROOT to the repository root")
os.environ["METAFL_ROOT"] = str(_repo)
os.chdir(_repo)

def _run_step(relative_path):
    _path = _repo / "pipeline_steps" / relative_path
    _code = compile(_path.read_text(encoding="utf-8"), str(_path), "exec")
    exec(_code, get_ipython().user_ns)
_run_step("main_01_setup_load.py")
_run_step("main_02_demographics_hc_reference.py")


MetaFL package root: .
Main-analysis output root: ./outputs/main_analyses
Cortical IDs: [3, 4, 5, 6, 19, 20, 21, 22, 25, 26, 31, 32]
Limbic IDs: [41, 42, 43, 44, 45, 46, 75, 76, 77, 78, 79, 80, 151, 152]
Cross-edge family: 168


,cohort,clinical_n,ses01_complete_n
0,HC,23,23
1,nTRD,25,25
2,TRD1,35,35
3,TRD2,104,104


,cohort,baseline_n,metric
0,nTRD,25,age/sex-adjusted normative-SC MwC deviation z
1,TRD1,35,age/sex-adjusted normative-SC MwC deviation z
2,TRD2,104,age/sex-adjusted normative-SC MwC deviation z


Written: ./outputs/main_analyses/demographic_and_clinical_summary.xlsx


,variable,HC,nTRD,TRD1,TRD2,primary_test,primary_p_value,TRD1_vs_TRD2_test,TRD1_vs_TRD2_p_value
0,Age,50.91 (17.04); n=23,52.32 (12.22); n=25,48.20 (15.52); n=35,39.09 (13.96); n=104,"one-way ANOVA: HC, nTRD, TRD1",0.561396,Welch t-test,0.003261
1,"Female, n (%)",14 (60.87%); n=23,14 (56.00%); n=25,25 (71.43%); n=35,73 (70.19%); n=104,"Pearson chi-square: HC, nTRD, TRD1",0.443311,Fisher exact test,1.000000
2,Education,14.17 (3.46); n=23,13.44 (3.86); n=25,13.14 (3.83); n=35,14.57 (2.81); n=104,"one-way ANOVA: HC, nTRD, TRD1",0.587829,Welch t-test,0.048544


,variable,nTRD,TRD1,TRD2,primary_test,primary_p_value,TRD1_vs_TRD2_test,TRD1_vs_TRD2_p_value
0,Baseline HDRS-17,3.72 (3.22); n=25,15.89 (7.38); n=35,22.84 (5.11); n=104,Welch t-test: nTRD versus TRD1,1.726193e-11,Welch t-test,0.000005


,variable,note
0,MSM scores,Not required for the released analyses


,variable,iTBS,rTMS,sham,test,statistic,df_1,df_2,p_value
0,HDRS reduction (baseline - post),6.73 (8.20); n=37,7.65 (6.95); n=34,6.06 (4.94); n=33,ordinary one-way ANOVA,0.448348,2,101,0.639948


,variable,rTMS,iTBS,sham,test,p_value
0,"Age, years",39.35 (11.79); n=34,37.95 (14.52); n=37,40.09 (15.62); n=33,one-way ANOVA,0.809492
1,"Female, n (%)",23 (67.65%); n=34,27 (72.97%); n=37,23 (69.70%); n=33,Pearson chi-square,0.884317
2,"Education, years",14.53 (2.60); n=34,14.27 (2.59); n=37,14.94 (3.27); n=33,one-way ANOVA,0.611602
3,Baseline HDRS-17,22.74 (4.69); n=34,23.41 (5.53); n=37,22.30 (5.12); n=33,one-way ANOVA,0.663926
4,HDRS-17 reduction,7.65 (6.95); n=34,6.73 (8.20); n=37,6.06 (4.94); n=33,one-way ANOVA,0.639948


## Preamble


In [2]:
"""Auto-extracted pipeline step: Main 03 — baseline discovery and HDRS-17 sensitivity.

Do not edit generated copies independently of the source notebooks.
"""


'Auto-extracted pipeline step: Main 03 — baseline discovery and HDRS-17 sensitivity.\n\nDo not edit generated copies independently of the source notebooks.\n'

## Source 15


In [3]:
TRD1_OUT = OUTPUT_ROOT / "01_TRD1_discovery_agesex"
trd1_result = run_trd1_vs_ntrd(
    reference_covariates=["age", "sex"],
    glm_covariates=["age", "sex"],
    output_dir=TRD1_OUT,
)
trd1_counts = {
    "edge_uncorrected_p_lt_0p01": len(trd1_result["edge_reported"]),
    "regional_uncorrected_p_lt_0p01": len(trd1_result["regional_reported"]),
}
trd1_counts


{'edge_uncorrected_p_lt_0p01': 22, 'regional_uncorrected_p_lt_0p01': 7}

## Source 17


In [4]:
SENSITIVITY_OUT = OUTPUT_ROOT / "08_TRD1_vs_nTRD_HDRS_sensitivity"
SENSITIVITY_OUT.mkdir(parents=True, exist_ok=True)
MWC_DEVIATION_FILE = MWC_DEVIATION_INPUT_FILE


def run_hdrs_sensitivity(trd1_context):
    case = trd1_context["case_rows"]
    ntrd = trd1_context["ntrd"]
    keys = list(case["subject_key"]) + list(ntrd["subject_key"])
    group = np.array([1] * len(case) + [0] * len(ntrd), dtype=float)
    covariates = clinical.set_index("subject_key")[[
        "age", "sex", "hdrs_pre"
    ]]

    def analyze(values, labels, modality):
        rows = []
        for feature_index, label in enumerate(labels):
            frame = pd.DataFrame({
                "subject_key": keys,
                "group": group,
                "y": values[:, feature_index],
            }).join(covariates, on="subject_key")
            primary = fit_group_glm(frame, "y", ["age", "sex"])
            adjusted = fit_group_glm(
                frame, "y", ["age", "sex", "hdrs_pre"]
            )
            row = {
                "modality": modality,
                "TRD1_deviation": float(frame.loc[frame["group"] == 1, "y"].mean()),
                "nTRD_deviation": float(frame.loc[frame["group"] == 0, "y"].mean()),
                "primary_t_group": primary["t"],
                "primary_effect_size_partial_r": primary["partial_r"],
                "primary_p_uncorrected": primary["p"],
                "primary_n": primary["n"],
                "HDRS_adjusted_t_group": adjusted["t"],
                "HDRS_adjusted_effect_size_partial_r": adjusted["partial_r"],
                "HDRS_adjusted_p_uncorrected": adjusted["p"],
                "HDRS_adjusted_n": adjusted["n"],
            }
            if modality == "edge":
                row.update({"ROI_1": label[0], "ROI_2": label[1]})
            else:
                row.update({"ROI": label})
            rows.append(row)
        result = pd.DataFrame(rows)
        result["primary_q_fdr_BH_audit_only"] = bh_fdr(result["primary_p_uncorrected"])
        result["HDRS_adjusted_q_fdr_BH_audit_only"] = bh_fdr(
            result["HDRS_adjusted_p_uncorrected"]
        )
        result["primary_p_lt_0p01"] = result["primary_p_uncorrected"] < GROUP_P_THRESHOLD
        result["HDRS_adjusted_p_lt_0p01"] = (
            result["HDRS_adjusted_p_uncorrected"] < GROUP_P_THRESHOLD
        )
        result["selection_change"] = np.select(
            [
                result["primary_p_lt_0p01"] & result["HDRS_adjusted_p_lt_0p01"],
                result["primary_p_lt_0p01"] & ~result["HDRS_adjusted_p_lt_0p01"],
                ~result["primary_p_lt_0p01"] & result["HDRS_adjusted_p_lt_0p01"],
            ],
            ["retained", "lost_after_HDRS_adjustment", "new_after_HDRS_adjustment"],
            default="not_selected",
        )
        return result.sort_values("HDRS_adjusted_p_uncorrected").reset_index(drop=True)

    edge_values = np.vstack([trd1_context["case_edge"], trd1_context["ntrd_edge"]])
    regional_values = np.vstack([
        trd1_context["case_regional"].to_numpy(float),
        trd1_context["ntrd_regional"].to_numpy(float),
    ])
    mwc_source = pd.read_csv(MWC_DEVIATION_FILE)
    mwc_source["subject_key"] = mwc_source["subject_key"].map(norm_key)
    mwc_source = mwc_source.loc[
        mwc_source["session"].eq("ses-01")
        & mwc_source["analysis_cohort"].isin(["nTRD", "TRD1"])
    ].copy()
    assert not mwc_source["subject_key"].duplicated().any()
    assert mwc_source["analysis_cohort"].value_counts().to_dict() == {"TRD1": 35, "nTRD": 25}
    mwc_feature_columns = [c for c in mwc_source if c.startswith("MwC_deviation_")]
    assert len(mwc_feature_columns) == 26
    mwc_aligned = mwc_source.set_index("subject_key").loc[keys]
    assert np.array_equal(
        mwc_aligned["analysis_cohort"].to_numpy(),
        np.where(group == 1, "TRD1", "nTRD"),
    )
    mwc_values = mwc_aligned[mwc_feature_columns].to_numpy(float)
    mwc_labels = [column.removeprefix("MwC_deviation_") for column in mwc_feature_columns]

    return (
        analyze(edge_values, CROSS_EDGE_NAMES, "edge"),
        analyze(regional_values, ROI_COLUMNS, "regional"),
        analyze(mwc_values, mwc_labels, "MwC"),
    )


(sensitivity_edge_all, sensitivity_regional_all, sensitivity_mwc_all) = run_hdrs_sensitivity(trd1_result)
sensitivity_edge_p001 = sensitivity_edge_all[sensitivity_edge_all["HDRS_adjusted_p_lt_0p01"]].copy()
sensitivity_regional_p001 = sensitivity_regional_all[sensitivity_regional_all["HDRS_adjusted_p_lt_0p01"]].copy()
sensitivity_mwc_p001 = sensitivity_mwc_all[sensitivity_mwc_all["HDRS_adjusted_p_lt_0p01"]].copy()


def sensitivity_summary_row(modality, result_all, result_p001):
    return {
        "modality": modality,
        "tested_n": len(result_all),
        "primary_p_lt_0p01": int(result_all["primary_p_lt_0p01"].sum()),
        "HDRS_adjusted_p_lt_0p01": len(result_p001),
        "retained": int((result_all["selection_change"] == "retained").sum()),
        "lost": int((result_all["selection_change"] == "lost_after_HDRS_adjustment").sum()),
        "new": int((result_all["selection_change"] == "new_after_HDRS_adjustment").sum()),
    }


sensitivity_summary = pd.DataFrame([
    sensitivity_summary_row("edge", sensitivity_edge_all, sensitivity_edge_p001),
    sensitivity_summary_row("regional", sensitivity_regional_all, sensitivity_regional_p001),
    sensitivity_summary_row("MwC", sensitivity_mwc_all, sensitivity_mwc_p001),
])

sensitivity_file = SENSITIVITY_OUT / "TRD1_vs_nTRD_baseline_HDRS_sensitivity.xlsx"
with pd.ExcelWriter(sensitivity_file, engine="openpyxl") as writer:
    sensitivity_summary.to_excel(writer, sheet_name="summary", index=False)
    sensitivity_edge_all.to_excel(writer, sheet_name="edge_all", index=False)
    sensitivity_edge_p001.to_excel(writer, sheet_name="edge_HDRS_adjusted_p001", index=False)
    sensitivity_regional_all.to_excel(writer, sheet_name="regional_all", index=False)
    sensitivity_regional_p001.to_excel(writer, sheet_name="region_HDRS_adjusted_p001", index=False)
    sensitivity_mwc_all.to_excel(writer, sheet_name="MwC_all", index=False)
    sensitivity_mwc_p001.to_excel(writer, sheet_name="MwC_HDRS_adjusted_p001", index=False)
style_excel_workbook(sensitivity_file)
display(sensitivity_summary)


,modality,tested_n,primary_p_lt_0p01,HDRS_adjusted_p_lt_0p01,retained,lost,new
0,edge,168,22,2,2,20,0
1,regional,26,7,2,2,5,0
2,MwC,26,2,1,0,2,1


## Source 19


In [5]:
TRD2_OUT = OUTPUT_ROOT / "02_TRD2_deviation_prepost_agesex"
trd2_context = build_deviation_context(
    case_cohort="TRD2",
    reference_covariates=["age", "sex"],
    output_dir=TRD2_OUT,
)
print("TRD2 subject-level deviations:", trd2_context["case_edge"].shape, trd2_context["case_regional"].shape)


TRD2 subject-level deviations: (104, 168) (104, 26)


## Results preview

Key numerical tables and figures generated by this step are displayed below. The full-resolution files remain under `outputs/`.


In [6]:
from io import BytesIO
import json as _preview_json
import pandas as _preview_pd
from PIL import Image as PILImage
from IPython.display import Image as NotebookImage, Markdown, display

_inline_table_specs = [{'title': 'Regional SUVR findings (p < .01)', 'path': 'outputs/main_analyses/01_TRD1_discovery_agesex/regional_uncorrected_p_lt_0p01.xlsx'}, {'title': 'Metabolic-connectivity findings (p < .01)', 'path': 'outputs/main_analyses/01_TRD1_discovery_agesex/edge_uncorrected_p_lt_0p01.xlsx', 'max_rows': 30}, {'title': 'Baseline-severity sensitivity summary', 'path': 'outputs/main_analyses/08_TRD1_vs_nTRD_HDRS_sensitivity/TRD1_vs_nTRD_baseline_HDRS_sensitivity.xlsx', 'sheet': 'summary'}]
_inline_figure_paths = []

display(Markdown("### Key tables"))
for _table_spec in _inline_table_specs:
    _relative_table = _table_spec["path"]
    _table_path = _repo / _relative_table
    if not _table_path.is_file():
        raise FileNotFoundError(_table_path)
    if _table_path.suffix.lower() == ".csv":
        _preview_table = _preview_pd.read_csv(_table_path)
    elif _table_path.suffix.lower() in {".xlsx", ".xls"}:
        _preview_table = _preview_pd.read_excel(
            _table_path, sheet_name=_table_spec.get("sheet", 0)
        )
    elif _table_path.suffix.lower() == ".json":
        _payload = _preview_json.loads(_table_path.read_text(encoding="utf-8"))
        _checks = _payload.get("checks", _payload)
        _preview_table = _preview_pd.DataFrame(
            {"item": list(_checks), "value": list(_checks.values())}
        ) if isinstance(_checks, dict) else _preview_pd.json_normalize(_payload)
    else:
        raise ValueError(f"Unsupported table format: {_table_path.suffix}")
    if _table_spec.get("query"):
        _preview_table = _preview_table.query(_table_spec["query"]).reset_index(drop=True)
    _max_rows = int(_table_spec.get("max_rows", 40))
    display(Markdown(
        f"**{_table_spec['title']}**  \n"
        f"`{_relative_table}` — {len(_preview_table)} rows × {len(_preview_table.columns)} columns"
    ))
    with _preview_pd.option_context("display.max_columns", 40, "display.max_colwidth", 60):
        display(_preview_table.head(_max_rows))

if _inline_figure_paths:
    display(Markdown("### Figures"))
for _relative_figure in _inline_figure_paths:
    _figure_path = _repo / _relative_figure
    if not _figure_path.is_file():
        raise FileNotFoundError(_figure_path)
    with PILImage.open(_figure_path) as _source_image:
        _preview_image = _source_image.convert("RGB")
        _preview_image.thumbnail((1600, 1200), PILImage.Resampling.LANCZOS)
        _preview_buffer = BytesIO()
        _preview_image.save(_preview_buffer, format="PNG", optimize=True)
    display(Markdown(f"**`{_relative_figure}`**"))
    display(NotebookImage(data=_preview_buffer.getvalue(), format="png"))


### Key tables

**Regional SUVR findings (p < .01)**  
`outputs/main_analyses/01_TRD1_discovery_agesex/regional_uncorrected_p_lt_0p01.xlsx` — 7 rows × 9 columns

,ROI,TRD1_deviation,nTRD_deviation,t_group,effect_size_partial_r,p_group_uncorrected,n_used,q_fdr_BH_26_regions_audit_only,selected_uncorrected_p_lt_0p01
0,AAL3_078_Putamen_R,2.003611,-0.074899,4.193229,0.488832,0.000099,60,0.002570,True
1,AAL3_079_Pallidum_L,1.311900,0.183160,3.836697,0.456232,0.000319,60,0.004147,True
2,AAL3_077_Putamen_L,1.664573,0.115234,3.425306,0.416198,0.001156,60,0.007825,True
3,AAL3_080_Pallidum_R,1.409577,0.150120,3.412018,0.414862,0.001204,60,0.007825,True
4,AAL3_046_Amygdala_R,1.073815,0.186467,3.209625,0.394178,0.002201,60,0.011447,True
5,AAL3_045_Amygdala_L,0.877322,0.227559,2.948895,0.366624,0.004646,60,0.020131,True
6,AAL3_019_Frontal_Sup_Medial_L,-1.491804,-0.423556,-2.708472,-0.340330,0.008949,60,0.033240,True


**Metabolic-connectivity findings (p < .01)**  
`outputs/main_analyses/01_TRD1_discovery_agesex/edge_uncorrected_p_lt_0p01.xlsx` — 22 rows × 10 columns

,ROI_1,ROI_2,TRD1_deviation,nTRD_deviation,t_group,effect_size_partial_r,p_group_uncorrected,n_used,q_fdr_BH_168_edges_audit_only,selected_uncorrected_p_lt_0p01
0,AAL3_019_Frontal_Sup_Medial_L,AAL3_079_Pallidum_L,-1.449757,0.014011,-3.322782,-0.405819,0.001575,60,0.062516,True
1,AAL3_005_Frontal_Mid_2_L,AAL3_043_ParaHippocampal_L,-1.085701,0.181615,-3.295518,-0.403032,0.001708,60,0.062516,True
2,AAL3_021_Frontal_Med_Orb_L,AAL3_080_Pallidum_R,-1.229816,0.233088,-3.259832,-0.399367,0.001899,60,0.062516,True
3,AAL3_006_Frontal_Mid_2_R,AAL3_079_Pallidum_L,-2.016003,-0.064277,-3.234260,-0.396728,0.002048,60,0.062516,True
4,AAL3_006_Frontal_Mid_2_R,AAL3_043_ParaHippocampal_L,-1.268528,0.005179,-3.230684,-0.396359,0.002069,60,0.062516,True
5,AAL3_003_Frontal_Sup_2_L,AAL3_079_Pallidum_L,-0.976198,0.295178,-3.170529,-0.390111,0.002468,60,0.062516,True
6,AAL3_005_Frontal_Mid_2_L,AAL3_079_Pallidum_L,-1.470969,0.302697,-3.146331,-0.387582,0.002648,60,0.062516,True
7,AAL3_021_Frontal_Med_Orb_L,AAL3_079_Pallidum_L,-1.235255,0.138456,-3.067304,-0.379263,0.003324,60,0.062516,True
8,AAL3_025_OFCmed_L,AAL3_043_ParaHippocampal_L,-1.136775,0.199760,-3.064694,-0.378986,0.003349,60,0.062516,True
9,AAL3_003_Frontal_Sup_2_L,AAL3_043_ParaHippocampal_L,-1.024904,0.060146,-3.004934,-0.372631,0.003969,60,0.066676,True


**Baseline-severity sensitivity summary**  
`outputs/main_analyses/08_TRD1_vs_nTRD_HDRS_sensitivity/TRD1_vs_nTRD_baseline_HDRS_sensitivity.xlsx` — 3 rows × 7 columns

,modality,tested_n,primary_p_lt_0p01,HDRS_adjusted_p_lt_0p01,retained,lost,new
0,edge,168,22,2,2,20,0
1,regional,26,7,2,2,5,0
2,MwC,26,2,1,0,2,1
